# E-Commerce Arbitrage Analysis

This notebook provides analysis tools for evaluating deal performance and optimizing the arbitrage system.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from utils.database import DatabaseManager
from utils.config import get_config

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Initialize
config = get_config('../config/config.yaml')
db = DatabaseManager(config.database_url)

## Load Deals Data

In [ ]:
# Load all active deals
with db.get_session() as session:
    from utils.database import Deal
    deals = session.query(Deal).filter(Deal.status == 'new').all()
    
# Convert to DataFrame
deals_data = [{
    'asin': d.asin,
    'title': d.product_title,
    'source': d.source,
    'source_price': d.source_price,
    'amazon_price': d.amazon_price,
    'profit': d.estimated_profit,
    'roi': d.roi_percentage,
    'sales_rank': d.sales_rank,
    'rating': d.average_rating,
    'review_count': d.review_count,
    'category': d.category,
    'first_seen': d.first_seen
} for d in deals]

df = pd.DataFrame(deals_data)
print(f"Loaded {len(df)} deals")
df.head()

## Deal Statistics

In [ ]:
# Summary statistics
print("=" * 60)
print("DEAL STATISTICS")
print("=" * 60)
print(f"\nTotal Deals: {len(df)}")
print(f"\nAverage ROI: {df['roi'].mean():.1f}%")
print(f"Median ROI: {df['roi'].median():.1f}%")
print(f"Max ROI: {df['roi'].max():.1f}%")
print(f"\nAverage Profit: ${df['profit'].mean():.2f}")
print(f"Median Profit: ${df['profit'].median():.2f}")
print(f"Total Potential Profit: ${df['profit'].sum():.2f}")
print(f"\nDeals by Source:")
print(df['source'].value_counts())
print("\n" + "=" * 60)

## ROI Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ROI histogram
axes[0].hist(df['roi'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('ROI (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('ROI Distribution')
axes[0].axvline(df['roi'].mean(), color='red', linestyle='--', label=f'Mean: {df["roi"].mean():.1f}%')
axes[0].legend()

# Profit histogram
axes[1].hist(df['profit'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Profit ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Profit Distribution')
axes[1].axvline(df['profit'].mean(), color='red', linestyle='--', label=f'Mean: ${df["profit"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## Category Analysis

In [ ]:
# Analyze by category
category_stats = df.groupby('category').agg({
    'roi': ['mean', 'median', 'count'],
    'profit': ['mean', 'sum']
}).round(2)

category_stats.columns = ['Avg ROI', 'Median ROI', 'Count', 'Avg Profit', 'Total Profit']
category_stats = category_stats.sort_values('Avg ROI', ascending=False)

print("\nCategory Performance:")
print(category_stats)

## Price vs Profit Relationship

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(df['source_price'], df['roi'], alpha=0.5)
plt.xlabel('Source Price ($)')
plt.ylabel('ROI (%)')
plt.title('Source Price vs ROI')
plt.grid(True, alpha=0.3)
plt.show()

# Calculate correlation
correlation = df[['source_price', 'roi', 'profit', 'sales_rank']].corr()
print("\nCorrelation Matrix:")
print(correlation)

## Top Opportunities

In [ ]:
# Top 10 by ROI
print("\nTop 10 Deals by ROI:")
top_roi = df.nlargest(10, 'roi')[['title', 'asin', 'source_price', 'amazon_price', 'profit', 'roi']]
print(top_roi.to_string(index=False))

# Top 10 by absolute profit
print("\n\nTop 10 Deals by Profit:")
top_profit = df.nlargest(10, 'profit')[['title', 'asin', 'source_price', 'amazon_price', 'profit', 'roi']]
print(top_profit.to_string(index=False))

## Sales Rank Analysis

In [ ]:
# Filter out deals without sales rank
df_ranked = df[df['sales_rank'] > 0]

if len(df_ranked) > 0:
    plt.figure(figsize=(12, 6))
    plt.scatter(df_ranked['sales_rank'], df_ranked['roi'], alpha=0.5)
    plt.xlabel('Sales Rank (lower is better)')
    plt.ylabel('ROI (%)')
    plt.title('Sales Rank vs ROI')
    plt.xscale('log')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Average ROI by sales rank buckets
    df_ranked['rank_bucket'] = pd.cut(df_ranked['sales_rank'], 
                                       bins=[0, 10000, 50000, 100000, 500000, 1000000],
                                       labels=['<10k', '10k-50k', '50k-100k', '100k-500k', '500k+'])
    
    rank_analysis = df_ranked.groupby('rank_bucket').agg({
        'roi': 'mean',
        'profit': 'mean',
        'asin': 'count'
    }).round(2)
    
    rank_analysis.columns = ['Avg ROI', 'Avg Profit', 'Count']
    print("\nPerformance by Sales Rank:")
    print(rank_analysis)
else:
    print("No sales rank data available")

## Deal Discovery Timeline

In [ ]:
# Deals over time
df['date'] = pd.to_datetime(df['first_seen']).dt.date
deals_by_date = df.groupby('date').size()

plt.figure(figsize=(12, 6))
deals_by_date.plot(kind='bar')
plt.xlabel('Date')
plt.ylabel('Number of Deals Found')
plt.title('Deals Discovered Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Recommendations

Based on the analysis, identify:
1. Best performing categories
2. Optimal price ranges
3. Sales rank thresholds
4. ROI vs velocity tradeoffs

In [ ]:
print("\n" + "=" * 60)
print("RECOMMENDATIONS")
print("=" * 60)

# Best categories
if len(category_stats) > 0:
    best_category = category_stats.index[0]
    print(f"\n1. Focus on '{best_category}' category (Avg ROI: {category_stats.iloc[0]['Avg ROI']:.1f}%)")

# Optimal price range
price_buckets = pd.cut(df['source_price'], bins=[0, 10, 25, 50, 100, 200])
price_analysis = df.groupby(price_buckets)['roi'].mean().sort_values(ascending=False)
best_price_range = price_analysis.index[0]
print(f"\n2. Best price range: ${best_price_range} (Avg ROI: {price_analysis.iloc[0]:.1f}%)")

# High velocity items (low sales rank, good ROI)
if len(df_ranked) > 0:
    high_velocity = df_ranked[(df_ranked['sales_rank'] < 30000) & (df_ranked['roi'] > 40)]
    print(f"\n3. {len(high_velocity)} high-velocity deals (rank < 30k, ROI > 40%)")

print("\n" + "=" * 60)